In [ ]:
import pandas as pd
import html
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
import random
import numpy as np
import spacy
from src.nlp_functions import(remove_geo_entities)

# Reproducibility 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

/Users/heather.iriye/Library/CloudStorage/OneDrive-KarolinskaInstitutet/Documents/GitHub/eurepoc_cyber_security-/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Read in data 
url = "https://zenodo.org/records/14965395/files/eurepoc_dyadic_dataset_0_1.csv?download=1"

dyadic_data_full = pd.read_csv(url)

print(dyadic_data_full.shape)
print(dyadic_data_full.head())

(4296, 57)
   dyad_id initiator_country initiator_alpha_2 receiver_country  \
0        0       Afghanistan                AF      Afghanistan   
1        1       Afghanistan                AF         Pakistan   
2        1       Afghanistan                AF         Pakistan   
3        2           Algeria                DZ          Algeria   
4        2           Algeria                DZ          Algeria   

  receiver_country_alpha_2_code  incident_id  \
0                            AF         4278   
1                            PK          488   
2                            PK          499   
3                            DZ          525   
4                            DZ         1183   

                                                name  \
0  TalibLeaks Hacked and Leaked Documents from Ta...   
1               Afghan Cyber Army attack on Pakistan   
2       Afghan Cyber Army attack on Pakistan Part II   
3            Over-X vs. Algerian ministry of housing   
4               N

In [3]:
# Isolate columns of interest
drop_vars = ["added_to_db","updated_at","attribution_id",\
    "initiator_alpha_2","receiver_country_alpha_2_code",
    "Data theft","Data theft & Doxing","Disruption",
    "Hijacking with Misuse","Hijacking without Misuse",
    "Ransomware","Not available"]

attack_labels = ["Data theft","Data theft & Doxing", \
    "Disruption","Hijacking with Misuse","Hijacking without Misuse",
    "Ransomware","Not available"]
attack_data = dyadic_data_full[["incident_id"] + attack_labels].copy()

dyadic_data = dyadic_data_full.drop(columns=drop_vars,\
    errors="ignore")

print(f"Original shape: {dyadic_data_full.shape}")
print(f"New shape: {dyadic_data.shape}")

print("\nRemaining columns:")
print(dyadic_data.columns.tolist())


Original shape: (4296, 57)
New shape: (4296, 45)

Remaining columns:
['dyad_id', 'initiator_country', 'receiver_country', 'incident_id', 'name', 'description', 'start_date', 'end_date', 'source_disclosure', 'operation_type', 'impact_indicator_score', 'impact_indicator_label', 'unweighted_intensity', 'weighted_intensity', 'number_attributions', 'number_political_responses', 'number_legal_responses', 'casualties', 'initiator_name', 'initiator_category', 'initiator_subcategory', 'receiver_id', 'receiver_name', 'receiver_category', 'receiver_subcategory', 'receiver_regions', 'offline_conflict_issue', 'offline_conflict_name', 'offline_conflict_intensity', 'offline_conflict_intensity_subcode', 'cyber_conflict_issue', 'physical_effects_spatial', 'physical_effects_temporal', 'target_multiplier', 'functional_impact', 'intelligence_impact', 'economic_impact', 'economic_impact_value', 'economic_impact_currency', 'affected_entities', 'affected_entities_value', 'affected_eu_countries', 'affected_eu

In [4]:
# Check for missing values in descriptions
print(dyadic_data["description"].isnull().sum())
print(dyadic_data[dyadic_data["description"]==""])

0
Empty DataFrame
Columns: [dyad_id, initiator_country, receiver_country, incident_id, name, description, start_date, end_date, source_disclosure, operation_type, impact_indicator_score, impact_indicator_label, unweighted_intensity, weighted_intensity, number_attributions, number_political_responses, number_legal_responses, casualties, initiator_name, initiator_category, initiator_subcategory, receiver_id, receiver_name, receiver_category, receiver_subcategory, receiver_regions, offline_conflict_issue, offline_conflict_name, offline_conflict_intensity, offline_conflict_intensity_subcode, cyber_conflict_issue, physical_effects_spatial, physical_effects_temporal, target_multiplier, functional_impact, intelligence_impact, economic_impact, economic_impact_value, economic_impact_currency, affected_entities, affected_entities_value, affected_eu_countries, affected_eu_countries_value, affected_third_countries, affected_third_countries_value]
Index: []

[0 rows x 45 columns]


In [5]:
# Incident-level dataset for NLP
# Descriptions can be duplicated because of the dyadic nature of the dataset. 
# One incident may span several rows if it affected several countries. 
incident_text_data = (
    dyadic_data
    .drop_duplicates(subset=["incident_id"])
    .copy()
)
print(incident_text_data.head())


   dyad_id initiator_country receiver_country  incident_id  \
0        0       Afghanistan      Afghanistan         4278   
1        1       Afghanistan         Pakistan          488   
2        1       Afghanistan         Pakistan          499   
3        2           Algeria          Algeria          525   
4        2           Algeria          Algeria         1183   

                                                name  \
0  TalibLeaks Hacked and Leaked Documents from Ta...   
1               Afghan Cyber Army attack on Pakistan   
2       Afghan Cyber Army attack on Pakistan Part II   
3            Over-X vs. Algerian ministry of housing   
4               North African Fox Espionage campaign   

                                         description           start_date  \
0  On 7 February 2025, a group of hackers, callin...  2024-01-01 00:00:00   
1  Afghan hackers deface six Pakistani government...  2013-07-11 00:00:00   
2  Afghan hackers hack the webpage of the Pakista...  2013-

In [6]:
# Clean description
incident_text_data["description"] = (
    incident_text_data["description"] 
    .fillna("")
    .astype(str)
    .apply(html.unescape)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [7]:
# Identify missing descriptions
print(incident_text_data["description"].isna().sum())

# Empty descriptions
print((incident_text_data["description"] == "").sum())

# Length distribution
incident_text_data["n_words"] = (
    incident_text_data["description"]
    .str.split()
    .str.len()
)

print(incident_text_data["n_words"].describe())

incident_text_data.loc[
    incident_text_data["n_words"] < 5,
    ["incident_id", "description"]
].head(20)


0
0
count    2957.000000
mean       80.144065
std        69.850175
min         3.000000
25%        29.000000
50%        65.000000
75%       112.000000
max       975.000000
Name: n_words, dtype: float64


,incident_id,description
1366,361,Taliban website hacked
2433,370,Israeli Government Site Hacked
4058,358,The Unknowns' hack NASA


In [8]:
# Check how many short descriptions would be lost if filtered out
print((incident_text_data["n_words"] < 5).sum())
print((incident_text_data["n_words"] < 10).sum())

3
105


In [9]:
# Filter out descriptions with less than five words
incident_text_data = incident_text_data[incident_text_data["n_words"] > 5]

In [10]:
# Remove geographic terms from text
nlp = spacy.load("en_core_web_sm")

incident_text_data["description_no_geo"] = (
    incident_text_data["description"]
    .apply(lambda x: remove_geo_entities(x, nlp))
)

In [11]:
# Sentence embeddings with BERT
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = model.encode(
    incident_text_data["description_no_geo"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 92/92 [00:04<00:00, 20.17it/s]


In [12]:
# Generic stopwords 
cyber_stopwords = [
    "attack",
    "attacks",
    "attacker",
    "hack",
    "hacks",
    "hacked",
    "hacker",
    "hackers",
    "hacking",
    "campaign",
    "group",
    "security",
    "incident",
    "reported",
    "report",
    "unknown"
]

vectorizer_model = CountVectorizer(
    stop_words=cyber_stopwords,
    min_df=5,
    ngram_range=(1, 3)
)

In [13]:
# Get rid of stopwords, BERT wasn't using them effectively
#vectorizer_model = CountVectorizer(
#    stop_words="english",
#    min_df=5,
#    ngram_range=(1, 3)
#)

# BERTopic analysis 
# For reproducibility 
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
) 

# For more readable labels
representation_model = KeyBERTInspired()


# Build topic model 
topic_model = BERTopic(
    embedding_model=model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    min_topic_size=20,
    calculate_probabilities=True
)


topics, probs = topic_model.fit_transform(
    incident_text_data["description_no_geo"],
    embeddings
)

topic_model = topic_model.reduce_topics(
    incident_text_data["description_no_geo"],
    nr_topics=10
)

topics = topic_model.topics_

incident_text_data["topic"] = topics
incident_text_data["topic_prob"] = probs.max(axis=1)

In [ ]:
# Save model 
topic_model.save("eurepoc_final_model")

incident_text_data[
    ["incident_id", "topic", "topic_prob"]
].to_csv(
    "eurepoc_topic_assignments.csv",
    index=False
) 

topic_model.get_topic_info().to_csv(
    "topic_info.csv",
    index=False
)